# NeuroScan: train one model on Colab

Trains one model from a config and writes a **run bundle** to Google Drive. No MLflow here: the bundle is
imported into MLflow on the tracking machine afterwards.

**Before running**
1. On the tracking machine: `make colab-package` and upload `dist/neuroscan-<commit>.tar.gz` to
   `MyDrive/neuroscan/code/` in Google Drive.
2. In Colab, open 🔑 **Secrets** (left sidebar), add `KAGGLE_API_TOKEN` with your Kaggle token, and enable
   notebook access.
3. **Runtime → Change runtime type → GPU.**

If the session disconnects, run all cells again with the same settings: training resumes from the last
finished epoch.

In [ ]:
# Settings
PACKAGE = "neuroscan-XXXXXXX.tar.gz"  # file name in MyDrive/neuroscan/code/
CONFIG = "configs/baseline_cnn.yaml"  # baseline_cnn | cnn_gap | efficientnet_b0 | resnet50
DRY_RUN = True                        # 1 epoch per stage on 32 images, to test the pipeline

DRIVE_ROOT = "/content/drive/MyDrive/neuroscan"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: change the runtime type"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Install the code

In [ ]:
!rm -rf /content/neuroscan
!tar -xzf "{DRIVE_ROOT}/code/{PACKAGE}" -C /content
%cd /content/neuroscan
!cat GIT_COMMIT
!pip install -q -e .
!python -c "import tensorflow as tf, keras, neuroscan; print('TF', tf.__version__, '| Keras', keras.__version__, '| neuroscan', neuroscan.__version__)"

## Download the dataset

In [ ]:
import os

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!python -m neuroscan.data.download --out /content/data/raw

## Train

In [ ]:
flags = "--dry-run" if DRY_RUN else ""
!python -m neuroscan.train --config {CONFIG} --raw-dir /content/data/raw --out {DRIVE_ROOT}/runs {flags}

In [ ]:
!ls -la {DRIVE_ROOT}/runs/*

## Next: import on the tracking machine

1. Download the run folder from `MyDrive/neuroscan/runs/<run-name>/` into `runs/` in the repository.
2. Run `make import BUNDLE=runs/<run-name>` and open `make mlflow`.